This notebook generates the **signal features** used for the modeling stage from cleaned OHLCV data.

It computes return-based, volatility, trend, and candle-structure indicators, and prepares a feature matrix suitable for downstream training and backtesting.

In [2]:
# imports
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath(".."))
from src.utils.data_loader import DataLoader



In [3]:
# import a dataset to test with
data_loader = DataLoader()
df = data_loader.load_single_data("clean", "AAPL")
df.head()

AAPL.csv loaded succesfully


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800


In [4]:
# calculate returns and log returns and indicators related to log returns
df["Returns"] = df["Adj Close"].pct_change()
# regular returns will be removed for training, they are there as visual aid
df["Log Returns"] = np.log(df["Adj Close"]).diff()
df["Monthly Log Returns"] = df["Log Returns"].rolling(21).sum()
df["Quarterly Log Returns"] = df["Log Returns"].rolling(63).sum()
df["Semi-annual Log Returns"] = df["Log Returns"].rolling(126).sum()

df



,Adj Close,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns
Date,,,,,,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600,NaN,NaN,NaN,NaN,NaN
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800,0.001729,0.001727,NaN,NaN,NaN
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000,-0.015906,-0.016034,NaN,NaN,NaN
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200,-0.001849,-0.001850,NaN,NaN,NaN
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800,0.006648,0.006626,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,255.171234,255.410004,256.559998,249.800003,251.479996,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558
2026-01-27,258.028534,258.269989,261.950012,258.209991,259.170013,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899
2026-01-28,256.200287,256.440002,258.859985,254.509995,257.649994,41288000,-0.007085,-0.007111,-0.064041,-0.046141,0.195861


In [5]:
# Daily volatility and rolling standard deviaton
df["Daily Volatility"] = np.sqrt( (1 / (4*np.log(2))) * ( (np.log( df["High"]/ df["Low"])) **2 ))
# use parkinson estimator for daily volatility
df["Monthly Volatility"] = df["Log Returns"].rolling(21).std()
df["Quarterly Volatility"] = df["Log Returns"].rolling(63).std()
df["Semi-annual Volatility"] = df["Log Returns"].rolling(126).std()
# uses n - 1 in denominator for sample variance
df["Volatility Ratio"] = df["Monthly Volatility"] / df["Quarterly Volatility"]

df





,Adj Close,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,Semi-annual Log Returns,Daily Volatility,Monthly Volatility,Quarterly Volatility,Semi-annual Volatility,Volatility Ratio
Date,,,,,,,,,,,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600,NaN,NaN,NaN,NaN,NaN,0.005965,NaN,NaN,NaN,NaN
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800,0.001729,0.001727,NaN,NaN,NaN,0.006554,NaN,NaN,NaN,NaN
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000,-0.015906,-0.016034,NaN,NaN,NaN,0.012633,NaN,NaN,NaN,NaN
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200,-0.001849,-0.001850,NaN,NaN,NaN,0.008416,NaN,NaN,NaN,NaN
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800,0.006648,0.006626,NaN,NaN,NaN,0.008387,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,255.171234,255.410004,256.559998,249.800003,251.479996,55969200,0.029713,0.029280,-0.064255,-0.015226,0.179558,0.016036,0.011845,0.010289,0.014020,1.151159
2026-01-27,258.028534,258.269989,261.950012,258.209991,259.170013,49648300,0.011198,0.011135,-0.058429,-0.016495,0.189899,0.008636,0.012116,0.010266,0.014047,1.180237
2026-01-28,256.200287,256.440002,258.859985,254.509995,257.649994,41288000,-0.007085,-0.007111,-0.064041,-0.046141,0.195861,0.010178,0.012148,0.009876,0.014007,1.230073


In [6]:
# movering averages
df["MA20"] = df["Log Returns"].rolling(20).mean()
df["MA60"] = df["Log Returns"].rolling(60).mean()
df["MA20/MA60"] = df["MA20"] / df["MA60"]
df["Distance from MA20"] = ((df["Adj Close"] - df["MA20"]) / df["MA20"] ) * 100
df["Distance from MA60"] = ((df["Adj Close"] - df["MA20"]) / df["MA60"] ) * 100


df

,Adj Close,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,...,Daily Volatility,Monthly Volatility,Quarterly Volatility,Semi-annual Volatility,Volatility Ratio,MA20,MA60,MA20/MA60,Distance from MA20,Distance from MA60
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600,NaN,NaN,NaN,NaN,...,0.005965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800,0.001729,0.001727,NaN,NaN,...,0.006554,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000,-0.015906,-0.016034,NaN,NaN,...,0.012633,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200,-0.001849,-0.001850,NaN,NaN,...,0.008416,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800,0.006648,0.006626,NaN,NaN,...,0.008387,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,255.171234,255.410004,256.559998,249.800003,251.479996,55969200,0.029713,0.029280,-0.064255,-0.015226,...,0.016036,0.011845,0.010289,0.014020,1.151159,-0.003478,-0.000848,4.102286,-7.336372e+06,-3.009590e+07
2026-01-27,258.028534,258.269989,261.950012,258.209991,259.170013,49648300,0.011198,0.011135,-0.058429,-0.016495,...,0.008636,0.012116,0.010266,0.014047,1.180237,-0.002847,-0.000706,4.034203,-9.064793e+06,-3.656922e+07
2026-01-28,256.200287,256.440002,258.859985,254.509995,257.649994,41288000,-0.007085,-0.007111,-0.064041,-0.046141,...,0.010178,0.012148,0.009876,0.014007,1.230073,-0.003268,-0.000929,3.518238,-7.840118e+06,-2.758340e+07


In [7]:
# volume averages and ratios
df["VMA20"] = df["Volume"].rolling(20).mean()
df["VMA60"] = df["Volume"].rolling(60).mean()
df["Relative Volume"] = df["Volume"] / df["VMA20"].shift(1)
# use 20 days preceding today to avoid self contamination in denominator
df["Volume Flow Ratio"] = df["VMA20"] / df["VMA60"]
df

,Adj Close,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,...,Volatility Ratio,MA20,MA60,MA20/MA60,Distance from MA20,Distance from MA60,VMA20,VMA60,Relative Volume,Volume Flow Ratio
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800,0.001729,0.001727,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000,-0.015906,-0.016034,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200,-0.001849,-0.001850,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800,0.006648,0.006626,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,255.171234,255.410004,256.559998,249.800003,251.479996,55969200,0.029713,0.029280,-0.064255,-0.015226,...,1.151159,-0.003478,-0.000848,4.102286,-7.336372e+06,-3.009590e+07,44202740.0,4.664386e+07,1.323155,0.947665
2026-01-27,258.028534,258.269989,261.950012,258.209991,259.170013,49648300,0.011198,0.011135,-0.058429,-0.016495,...,1.180237,-0.002847,-0.000706,4.034203,-9.064793e+06,-3.656922e+07,45609065.0,4.661989e+07,1.123195,0.978318
2026-01-28,256.200287,256.440002,258.859985,254.509995,257.649994,41288000,-0.007085,-0.007111,-0.064041,-0.046141,...,1.230073,-0.003268,-0.000929,3.518238,-7.840118e+06,-2.758340e+07,46487705.0,4.614325e+07,0.905259,1.007465


In [8]:
# microstructure
df["Daily Range"] = df["High"] - df["Low"]
# wont be used as indicator, just there to help next calculations
df["Monthly Average Intraday Range"] = df["Daily Range"].rolling(21).mean()
df["True Range"] = np.maximum.reduce([df["Daily Range"],(df["High"] - df["Adj Close"].shift(1)).abs(),(df["Low"] - df["Adj Close"].shift(1)).abs()])
# wont be used as indicator, just there to help next calculations
df["Monthly Average True Range"] = df["True Range"].rolling(21).mean()
df["Daily Candle Body"] = (df["Adj Close"] - df["Open"]).abs()
# wont be used as indicator, just there to help next calculations
df["Monthly Average Candle Body"] = df["Daily Candle Body"].rolling(21).mean()
df["Percentage Intraday Range"] = ((df["High"] - df["Low"]) / df["Open"]) * 100
df["Monthly Average Percentage Intraday Range"] = df["Percentage Intraday Range"].rolling(21).mean()


df



,Adj Close,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,...,Relative Volume,Volume Flow Ratio,Daily Range,Monthly Average Intraday Range,True Range,Monthly Average True Range,Daily Candle Body,Monthly Average Candle Body,Percentage Intraday Range,Monthly Average Percentage Intraday Range
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600,NaN,NaN,NaN,NaN,...,NaN,NaN,0.075714,NaN,NaN,NaN,1.210117,NaN,0.993298,NaN
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800,0.001729,0.001727,NaN,NaN,...,NaN,NaN,0.083572,NaN,1.287260,NaN,1.240817,NaN,1.090407,NaN
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000,-0.015906,-0.016034,NaN,NaN,...,NaN,NaN,0.160000,NaN,1.263317,NaN,1.335133,NaN,2.089751,NaN
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200,-0.001849,-0.001850,NaN,NaN,...,NaN,NaN,0.105358,NaN,1.250133,NaN,1.252889,NaN,1.393159,NaN
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800,0.006648,0.006626,NaN,NaN,...,NaN,NaN,0.105000,NaN,1.261818,NaN,1.159154,NaN,1.397997,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,255.171234,255.410004,256.559998,249.800003,251.479996,55969200,0.029713,0.029280,-0.064255,-0.015226,...,1.323155,0.947665,6.759995,4.554760,8.751877,4.750748,3.691238,1.939210,2.688084,1.751588
2026-01-27,258.028534,258.269989,261.950012,258.209991,259.170013,49648300,0.011198,0.011135,-0.058429,-0.016495,...,1.123195,0.978318,3.740021,4.579048,6.778778,4.915232,1.141479,1.935756,1.443076,1.763830
2026-01-28,256.200287,256.440002,258.859985,254.509995,257.649994,41288000,-0.007085,-0.007111,-0.064041,-0.046141,...,0.905259,1.007465,4.349991,4.666666,4.349991,5.002850,1.449707,1.956428,1.688333,1.800630


In [9]:
# technical indicators
# RSI
df["Price Difference"] = df["Adj Close"].diff()
df["Gains"] = df["Price Difference"].where(df["Price Difference"] > 0, 0)
df["Losses"] = df["Price Difference"].where(df["Price Difference"] < 0, 0).abs()
df["Average Gains"] = df["Gains"].rolling(14).mean()
df["Average Losses"] = df["Losses"].rolling(14).mean()
df["Average Gains"] = df["Gains"].ewm(alpha=1/14, adjust=False).mean()
df["Average Losses"] = df["Losses"].ewm(alpha=1/14, adjust=False).mean()
df["RS"] = df["Average Gains"] / df["Average Losses"]
df["RSI"] = 100 - (100 / (1 + df["RS"]))

# ATR 14
df["14-day Average True Range"] = df["True Range"].ewm(alpha=1/14, adjust=False).mean()

# NATR
df["Normalized Average True Range"] = (df["14-day Average True Range"]/df["Adj Close"]) * 100


# Efficiency Ratio 10-days
df["10-day Efficiency Ratio"] = df["Adj Close"].diff(10).abs() / df["Adj Close"].diff().abs().rolling(10).sum()
# Efficiency Ratio 21-days
df["10-day Efficiency Ratio"] = df["Adj Close"].diff(21).abs() / df["Adj Close"].diff().abs().rolling(21).sum()

# Conviction Ratio
df["Body Up"] = df["Daily Candle Body"].clip(lower=0)
df["Body Down"] = -df["Daily Candle Body"].clip(upper=0)
df["Monthly Conviction Ratio"] = (df["Body Up"].rolling(21).mean() /df["Body Down"].rolling(21).mean())



df





,Adj Close,Close,High,Low,Open,Volume,Returns,Log Returns,Monthly Log Returns,Quarterly Log Returns,...,Average Gains,Average Losses,RS,RSI,14-day Average True Range,Normalized Average True Range,10-day Efficiency Ratio,Body Up,Body Down,Monthly Conviction Ratio
Date,,,,,,,,,,,,,,,,,,,,,
2010-01-04,6.412383,7.643214,7.660714,7.585000,7.622500,493729600,NaN,NaN,NaN,NaN,...,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,1.210117,-0.0,NaN
2010-01-05,6.423469,7.656429,7.699643,7.616071,7.664286,601904800,0.001729,0.001727,NaN,NaN,...,0.000792,0.000000,inf,100.000000,1.287260,20.039951,NaN,1.240817,-0.0,NaN
2010-01-06,6.321296,7.534643,7.686786,7.526786,7.656429,552160000,-0.015906,-0.016034,NaN,NaN,...,0.000735,0.007298,0.100752,9.153025,1.285550,20.336808,NaN,1.335133,-0.0,NaN
2010-01-07,6.309611,7.520714,7.571429,7.466071,7.562500,477131200,-0.001849,-0.001850,NaN,NaN,...,0.000683,0.007611,0.089704,8.231931,1.283020,20.334377,NaN,1.252889,-0.0,NaN
2010-01-08,6.351560,7.570714,7.571429,7.466429,7.510714,447610800,0.006648,0.006626,NaN,NaN,...,0.003630,0.007068,0.513648,33.934461,1.281506,20.176235,NaN,1.159154,-0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-01-26,255.171234,255.410004,256.559998,249.800003,251.479996,55969200,0.029713,0.029280,-0.064255,-0.015226,...,0.913798,1.352045,0.675863,40.329259,5.145254,2.016392,0.397607,3.691238,-0.0,-inf
2026-01-27,258.028534,258.269989,261.950012,258.209991,259.170013,49648300,0.011198,0.011135,-0.058429,-0.016495,...,1.052619,1.255471,0.838426,45.605643,5.261934,2.039284,0.352862,1.141479,-0.0,-inf
2026-01-28,256.200287,256.440002,258.859985,254.509995,257.649994,41288000,-0.007085,-0.007111,-0.064041,-0.046141,...,0.977432,1.296383,0.753969,42.986434,5.196795,2.028411,0.373075,1.449707,-0.0,-inf


In [10]:
df.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Returns',
       'Log Returns', 'Monthly Log Returns', 'Quarterly Log Returns',
       'Semi-annual Log Returns', 'Daily Volatility', 'Monthly Volatility',
       'Quarterly Volatility', 'Semi-annual Volatility', 'Volatility Ratio',
       'MA20', 'MA60', 'MA20/MA60', 'Distance from MA20', 'Distance from MA60',
       'VMA20', 'VMA60', 'Relative Volume', 'Volume Flow Ratio', 'Daily Range',
       'Monthly Average Intraday Range', 'True Range',
       'Monthly Average True Range', 'Daily Candle Body',
       'Monthly Average Candle Body', 'Percentage Intraday Range',
       'Monthly Average Percentage Intraday Range', 'Price Difference',
       'Gains', 'Losses', 'Average Gains', 'Average Losses', 'RS', 'RSI',
       '14-day Average True Range', 'Normalized Average True Range',
       '10-day Efficiency Ratio', 'Body Up', 'Body Down',
       'Monthly Conviction Ratio'],
      dtype='str')